In [17]:
import requests
import json
import pandas as pd
import mysql.connector
from keys import flights_key  




In [18]:
icao_airport = "RKPK"  # Busan
url = f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/RKPK/2025-09-01T08:00/2025-09-01T20:00"

querystring = {
    "offsetMinutes": "-120",
    "durationMinutes": "720",
    "withLeg": "true",
    "direction": "Both",
    "withCancelled": "true",
    "withCodeshared": "true",
    "withCargo": "true",
    "withPrivate": "true",
    "withLocation": "false"
}

headers = {
    "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
    "x-rapidapi-key": "cbae0c8b00msha53f5d91d17f5f2p1c7e20jsn6f6909a55841"  
}

response = requests.get(url, headers=headers, params=querystring)

if response.status_code != 200:
    raise Exception(f"Error API: {response.status_code}")

data = response.json()
print("Datos de la API obtenidos correctamente.")


Datos de la API obtenidos correctamente.


In [26]:
# Función para extraer información de cada vuelo
def flight_extraction(flight):
    record = {
        'scheduled_arrival_time': flight.get('arrival', {}).get('scheduledTime'),
        'flight_number': flight.get('number'),
        'from_airport': flight.get('departure', {}).get('airport', {}).get('name'),
        'airline': flight.get('airline', {}).get('name'),
        'aircraft': flight.get('aircraft', {}).get('model'),
        'icao_code': icao_airport  # agregamos el código ICAO
    }
    return pd.DataFrame([record])

# Crear DataFrame completo
flights_df = pd.concat(
    [flight_extraction(flight) for flight in data.get('arrivals', [])],
    ignore_index=True
)

# Extraer solo la hora local (sin zona horaria)
flights_df['scheduled_arrival_time'] = flights_df['scheduled_arrival_time'].apply(lambda x: x['local'])

# Convertir a datetime de pandas automáticamente
flights_df['scheduled_arrival_time'] = pd.to_datetime(flights_df['scheduled_arrival_time'], errors='coerce')

# Quitar la info de zona horaria (para que MySQL lo acepte)
flights_df['scheduled_arrival_time'] = flights_df['scheduled_arrival_time'].dt.tz_localize(None)

# Mostrar primeras filas
flights_df.head()



,scheduled_arrival_time,flight_number,from_airport,airline,aircraft,icao_code
0,2025-09-01 08:05:00,KE 1803,Seoul-si,Korean Air,Airbus A220-300,RKPK
1,2025-09-01 08:10:00,BX 602,Denpasar-Bali Island,Air Busan,Airbus A321 NEO,RKPK
2,2025-09-01 08:10:00,7C 2662,Singapore,Jeju Air,Boeing 737-800,RKPK
3,2025-09-01 08:30:00,7C 2552,Bangkok,Jeju Air,Boeing 737-800,RKPK
4,2025-09-01 08:40:00,VJ 990,Nha Trang,VietJetAir,Airbus A321,RKPK


In [20]:
import mysql.connector

connection = mysql.connector.connect(
    host="127.0.0.1",
    user="root",
    password="adricaba1107",
    database="gans"
)

cursor = connection.cursor()
print("Conexión establecida con MySQL.")



Conexión establecida con MySQL.


In [27]:
insert_query = """
INSERT INTO flight_arrival 
(scheduled_arrival_time, flight_number, from_airport, airline, aircraft, icao_code)
VALUES (%s, %s, %s, %s, %s, %s)
"""

for _, row in flights_df.iterrows():
    cursor.execute(insert_query, tuple(row))

connection.commit()
print("Datos insertados correctamente en la tabla flight_arrival.")


Datos insertados correctamente en la tabla flight_arrival.
